# CallGrader: grading AI voice-agent calls

CallGrader reads a phone call handled by an AI receptionist and answers five questions: did the call **pass**, did the caller get what they needed (**task success**), did the agent **miss a handoff** to a person or emergency line, did it **make something up**, and did it **resist a prompt injection**.

This notebook trains and compares three graders on the same 1,260 calls, on a free T4 GPU (Runtime > Change runtime type > T4 GPU, then Run all):

| Grader | What it is |
|---|---|
| Baseline | TF-IDF + logistic regression (scikit-learn) |
| Encoder | ModernBERT-base fine-tuned in PyTorch |
| LLM | Qwen2.5-0.5B-Instruct + LoRA, answering in JSON, served with vLLM |

Code, data and README: [github.com/adelsherif8/callgrader](https://github.com/adelsherif8/callgrader)

## 1. Setup
Check the GPU, download the project, install the libraries.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
%cd /content
if not os.path.exists("/content/callgrader"):
    !git clone -q https://github.com/adelsherif8/callgrader.git
%cd /content/callgrader
!pip -q install -U "transformers>=4.56" "peft>=0.17" scikit-learn
!pip -q uninstall -y torchao   # Colab's old torchao breaks peft's LoRA layers; not needed here
import importlib; importlib.invalidate_caches()
import torch, transformers, peft, sklearn
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(), "| transformers", transformers.__version__, "| peft", peft.__version__)

## 2. The data
1,260 simulated calls to three demo businesses (a plumber's callback line, a dental front desk, a self-storage receptionist), from the VoiceDesk eval harness. Labels are **silver**: code checks plus an LLM judge, not a human. Splits keep whole scenarios together, so test calls come from situations the models never saw.

In [ ]:
import pandas as pd
from callgrader.data import LABELS, read
rows = read("train") + read("val") + read("test")
df = pd.DataFrame([{"agent": r["agent"], "category": r["category"], "scenario": r["scenario"], **r["labels"]} for r in rows])
print(f"{len(df)} calls, {df.scenario.nunique()} scenarios, {df.category.nunique()} categories\n")
summary = pd.DataFrame({lab: {"applies to": int(df[lab].notna().sum()), "yes": int((df[lab] == True).sum())} for lab in LABELS}).T
summary["yes %"] = (100 * summary["yes"] / summary["applies to"]).round(1)
summary

In [ ]:
# One call as the grader reads it: the business fact card, then the transcript with tool calls
print(read("test")[0]["text"][:1500])

## 3. Baseline vs ModernBERT: 5-fold cross-validation
Every scenario is tested once, by a model trained without it. Three graders: always the most common answer (the floor), TF-IDF + logistic regression, and ModernBERT fine-tuned for 3 epochs per fold. About 55 minutes on a T4.

In [ ]:
import json
!python -m callgrader.cv --encoder 2>&1 | grep -E "'model'|^majority|^tfidf|^modernbert|Error"

In [ ]:
cv = json.load(open("results/cv.json"))
names = {"majority": "Always the common answer", "baseline": "TF-IDF + logistic regression", "encoder": "ModernBERT fine-tuned"}
table = pd.DataFrame([{"grader": names[k], **{lab: v["metrics"][lab]["f1"] for lab in LABELS},
                       "macro F1": v["metrics"]["macro_f1"], "95% CI": f'{v["macro_f1_95ci"][0]:.2f}-{v["macro_f1_95ci"][1]:.2f}',
                       "fold mean ± sd": f'{v["fold_mean"]:.2f} ± {v["fold_std"]:.2f}'} for k, v in cv.items()]).set_index("grader")
table

In [ ]:
# Save to Google Drive straight away: free Colab can reset the machine and wipe local files
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/callgrader-results && cp results/*.json /content/drive/MyDrive/callgrader-results/

## 4. Fine-tune an LLM with LoRA
Qwen2.5-0.5B-Instruct learns to answer each call with one JSON object (`true` / `false` / `null` per label). LoRA trains small add-on matrices next to the attention layers (under 1% of the weights); the loss is on the answer only. After training the adapter is merged back, so the model can be served as a normal one.

In [ ]:
import time
from callgrader import llm
t0 = time.perf_counter()
history = llm.train(epochs=2)
print(history, f"{(time.perf_counter() - t0) / 60:.1f} min")

In [ ]:
# Keep the merged model on Drive too (about 1 GB), so evaluation can resume after a reset
!mkdir -p /content/drive/MyDrive/callgrader-model && cp -r models/llm /content/drive/MyDrive/callgrader-model/

## 5. Evaluate the LLM on the test calls
Plain `transformers`, one call at a time, greedy decoding. Also reports how often the answer is valid JSON.

In [ ]:
res = {"model": llm.BASE + " + LoRA r16 (merged)", "history": history, "hardware": "Colab T4", **llm.evaluate()}
json.dump(res, open("results/llm.json", "w"), indent=2)
print(f'macro F1 {res["metrics"]["macro_f1"]} | {res["ms_per_call"]} ms per call | valid JSON {res["valid_json_rate"]:.0%}')
pd.DataFrame({lab: res["metrics"][lab] for lab in LABELS}).T

## 6. Serve with vLLM
vLLM keeps the GPU busy with continuous batching: all test calls go in at once and new ones join the batch as others finish. Same model, same prompts; this measures the speed-up over plain `transformers`.

In [ ]:
!pip -q install "vllm==0.11.*"

In [ ]:
import gc; gc.collect(); torch.cuda.empty_cache()
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from callgrader.metrics import report
tok = AutoTokenizer.from_pretrained("models/llm")
test = read("test")
prompts = [llm.prompt(tok, r["text"]) for r in test]
engine = LLM(model="models/llm", dtype="float16", max_model_len=2048, gpu_memory_utilization=0.6)
sp = SamplingParams(temperature=0, max_tokens=60)
engine.generate(prompts[:4], sp)                              # warm-up
t0 = time.perf_counter(); outs = engine.generate(prompts, sp); dt = time.perf_counter() - t0
preds = [llm.parse(o.outputs[0].text) for o in outs]
vres = {"engine": "vLLM (fp16, continuous batching)", "hardware": "Colab T4", "calls": len(test),
        "calls_per_second": round(len(test) / dt, 1), "ms_per_call": round(dt / len(test) * 1000, 1),
        "hf_ms_per_call": res["ms_per_call"], "speedup": round(res["ms_per_call"] / (dt / len(test) * 1000), 1),
        "metrics": report(test, preds, LABELS)}
json.dump(vres, open("results/vllm.json", "w"), indent=2)
print(f'{vres["calls_per_second"]} calls/s | {vres["ms_per_call"]} ms per call | {vres["speedup"]}x faster than transformers | macro F1 {vres["metrics"]["macro_f1"]}')

## 7. Save the results
Copies the result files to Google Drive (`MyDrive/callgrader-results`) so they survive the runtime ending.

In [ ]:
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/callgrader-results && cp results/*.json /content/drive/MyDrive/callgrader-results/ && ls /content/drive/MyDrive/callgrader-results